# Calibration

Calibration means finding the parameter values that make a model's output match observed data. This page does the shortest version of it:

1. run the model with a known contact rate to make some "observed" case counts (so we can check the answer);
2. say which model output the data describes (`Target`);
3. say what we believe about the parameter before seeing the data (a prior), and put model, prior and target together (`BayesianModel`);
4. ask for the single best-fitting value, the *maximum a posteriori* (MAP) estimate;
5. plot the fitted model output against the data.

The page runs in well under a minute.

## The model

An SIR model with one unknown parameter, `contact_rate`. The recovery rate is known and fixed at 0.2 per day. The population is 10,000 people, ten of them infectious at the start.

In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from summer4 import (
    FlowMass,
    FlowModel,
    OutputSet,
    Param,
    Property,
    PropertyMap,
    Target,
    TargetSet,
    TransitionFlow,
)
from summer4.epi import BayesianModel, ForceOfInfection, Poisson, Uniform

state = Property("state", ("S", "I", "R"))
model = FlowModel(PropertyMap.from_property(state))
model.add_flow(
    TransitionFlow(
        "infection",
        state["S"],
        state["I"],
        ForceOfInfection("foi", infectious=state["I"], contact_rate=Param("contact_rate")),
    )
)
model.add_flow(TransitionFlow("recovery", state["I"], state["R"], 0.2))
model.set_initial_population({state["S"]: 9990.0, state["I"]: 10.0})
cm = model.compile()

## The output the data describes

Data are usually counted events, such as new infections per day, not the number of people in a compartment. An `OutputSet` names the quantities we want from each run. Here `new_infections` is the infection flow, added up over the whole population (`sum_by()` with nothing in the brackets adds everything).

We then run the model with the true contact rate of 0.5, take the output on days 5, 10, ..., 40, and add random Poisson noise. Those noisy counts are our synthetic data. The plot shows the true curve and the data.

In [ ]:
run_kwargs = dict(t0=0.0, t1=60.0, dt=1.0)

outputs = OutputSet()
outputs["new_infections"] = FlowMass("infection").sum_by()

true_params = {"contact_rate": 0.5}
truth = outputs.evaluate(cm.run(true_params, save=outputs.plan(), **run_kwargs), true_params)[
    "new_infections"
]

obs_days = np.arange(5.0, 41.0, 5.0)
expected = np.asarray(truth.at_times(obs_days).values).reshape(-1)
observed = np.random.default_rng(1).poisson(expected).astype(float)

fig = pd.Series(
    np.asarray(truth.values).reshape(-1),
    index=pd.Index(np.asarray(truth.times.values), name="day"),
    name="true model",
).plot(
    title="Synthetic data and the model that made it",
    labels={"value": "new infections per day", "variable": ""},
)
fig.add_scatter(x=obs_days, y=observed, mode="markers", marker={"size": 9}, name="data")
fig.show()

## Target, prior and BayesianModel

A `Target` links the data to an output: its `key` is the output name, `times` and `values` are the observations, and `likelihood` says how the data scatter around the model (`Poisson` for counts).

A prior says what values of the parameter we consider reasonable before seeing the data. `Uniform("contact_rate", 0.1, 1.5)` says any value between 0.1 and 1.5 is equally likely.

`BayesianModel` puts the compiled model, the priors, the targets and the outputs together. Nothing is fitted yet; the cell shows what the model will use.

In [ ]:
target = Target(
    key="new_infections", times=obs_days, values=observed, likelihood=Poisson("sum")
)
bm = BayesianModel(
    cm,
    {},
    [Uniform("contact_rate", 0.1, 1.5)],
    TargetSet((target,)),
    outputs=outputs,
    run_kwargs=run_kwargs,
)
pd.Series({"prior": "contact_rate ~ Uniform(0.1, 1.5)", "target": target.key, "data points": len(observed)})

## Fitting

`find_map` searches for the parameter value with the highest posterior density, which is the best compromise between the prior and the data. It takes a number of optimisation steps and returns the best parameters found as `best_params`. The table compares the estimate with the true value of 0.5 we used to make the data.

In [ ]:
fit = bm.find_map(steps=200)
best = {name: float(value) for name, value in fit.best_params.items()}
pd.DataFrame({"true": true_params, "fitted": best})

## The fit against the data

Finally we run the model again with the fitted parameter and draw it over the data. The fitted line should pass close to the points.

In [ ]:
fitted = outputs.evaluate(cm.run(best, save=outputs.plan(), **run_kwargs), best)["new_infections"]

fig = pd.Series(
    np.asarray(fitted.values).reshape(-1),
    index=pd.Index(np.asarray(fitted.times.values), name="day"),
    name="fitted model",
).plot(
    title="Fitted model and data",
    labels={"value": "new infections per day", "variable": ""},
)
fig.add_scatter(x=obs_days, y=observed, mode="markers", marker={"size": 9}, name="data")
fig.show()

## Things to try

- Change the seed in `np.random.default_rng(1)` and refit. How far does the estimate move?
- Use only the first three observation days. Is the fit still good?
- Add a second unknown by replacing the fixed `0.2` in the recovery flow with `Param("recovery_rate")` and add a second prior, such as `Uniform("recovery_rate", 0.05, 0.5)`. Are both parameters recovered?
- Ask for a full set of plausible values instead of the single best one with `bm.sample(num_warmup=200, num_samples=200)`. The calibration notebooks in `examples/notebooks` show how to plot those draws.